# Variant E (GDN) Sub-100M on T4

Short-run setup for Kaggle T4 testing with checkpoint resume support.

In [ ]:
from pathlib import Path
import importlib
import importlib.util
import json
import os
import shlex
import subprocess
import sys

REQUIRED_PIP_PACKAGES = {
    'miditok': 'miditok',
    'pretty_midi': 'pretty_midi',
    'ncps': 'ncps',
    'safetensors': 'safetensors',
    'matplotlib': 'matplotlib',
    'huggingface_hub': 'huggingface_hub',
}

def ensure_runtime_dependencies() -> None:
    missing_specs = [
        spec
        for module_name, spec in REQUIRED_PIP_PACKAGES.items()
        if importlib.util.find_spec(module_name) is None
    ]
    if not missing_specs:
        print('Runtime dependencies ready.')
        return

    print(f'Installing missing package(s): {missing_specs}')
    try:
        subprocess.check_call(
            [
                sys.executable,
                '-m',
                'pip',
                'install',
                '--quiet',
                '--disable-pip-version-check',
                *missing_specs,
            ]
        )
    except subprocess.CalledProcessError as exc:
        raise RuntimeError(
            'Failed to install required packages. On Kaggle, enable Internet access or attach a dataset containing wheels.'
        ) from exc

    importlib.invalidate_caches()
    still_missing = [
        module_name
        for module_name in REQUIRED_PIP_PACKAGES
        if importlib.util.find_spec(module_name) is None
    ]
    if still_missing:
        raise ModuleNotFoundError(f'Required packages still missing after install: {still_missing}')
    print('Runtime dependencies installed.')

ensure_runtime_dependencies()

TRAIN_MODULE = 'training.train_variant_e_sub100m'
TRAIN_SCRIPT = Path('training/train_variant_e_sub100m.py')

def can_write_to(path: Path) -> bool:
    try:
        path.mkdir(parents=True, exist_ok=True)
        probe = path / '.copilot_write_probe'
        probe.write_text('ok', encoding='utf-8')
        probe.unlink()
        return True
    except Exception:
        return False

def find_project_root() -> Path:
    start = Path.cwd().resolve()
    probes = [start]
    if start.name.lower() == 'notebooks':
        probes.append(start.parent)

    kaggle_hints = [
        Path('/kaggle/working'),
        Path('/kaggle/input'),
        Path('/kaggle/working/piano_midi_model'),
        Path('/kaggle/input/piano_midi_model'),
        Path('/kaggle/input/piano-midi-model'),
        Path('/kaggle/input/midi-ai-piano-model'),
    ]
    probes.extend(path for path in kaggle_hints if path.exists())

    seen = set()
    for probe in probes:
        if not probe.exists():
            continue
        key = str(probe)
        if key in seen:
            continue
        seen.add(key)
        for parent in [probe, *probe.parents]:
            if (parent / TRAIN_SCRIPT).exists() and (parent / 'training' / '__init__.py').exists():
                return parent

    for scan_root in (Path('/kaggle/working'), Path('/kaggle/input')):
        if not scan_root.exists():
            continue
        hit = next(scan_root.rglob(str(TRAIN_SCRIPT).replace('\\', '/')), None)
        if hit is None:
            continue
        candidate = hit.parent.parent
        if (candidate / 'training' / '__init__.py').exists():
            return candidate

    raise FileNotFoundError(
        f'Could not locate project root containing {TRAIN_SCRIPT}. '
        'Copy the full repository into Kaggle input/working and retry.'
    )

def resolve_output_base(project_root: Path) -> Path:
    candidates = []
    kaggle_working = Path('/kaggle/working')
    if kaggle_working.exists():
        candidates.append(kaggle_working)
    candidates.append(project_root)

    for candidate in candidates:
        if can_write_to(candidate):
            return candidate

    raise OSError(
        'Could not find a writable output base. On Kaggle, keep code in /kaggle/input and write to /kaggle/working.'
    )

PROJECT_ROOT = find_project_root()
OUTPUT_BASE = resolve_output_base(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.environ['PYTHONPATH'] = str(PROJECT_ROOT) + os.pathsep + os.environ.get('PYTHONPATH', '')

print(f'Project root: {PROJECT_ROOT}')
print(f'Output base: {OUTPUT_BASE}')
print(f'Training module: {TRAIN_MODULE}')

In [ ]:
import numpy as np

OUTPUT_DIR = OUTPUT_BASE / 'outputs' / 'variant_e_40m_t4'
SIZE_PRESET = '40m'  # '40m' or '80m'
MAX_PIECES = 15000
EPOCHS = 6
BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 16
NUM_WORKERS = 2
LEARNING_RATE = 2e-4
SEED = 42
LOG_EVERY_N_STEPS = 20
RESUME_MODE = 'remaining'  # 'remaining' or 'additional'
AUTO_RESUME = True
ALLOW_FALLBACK_GDN = True
SEED_MIDI = ''  # optional

FORCE_REBUILD_MANIFEST = False
PRETOKENIZED_ROOT_CANDIDATES = [
    PROJECT_ROOT / 'processed',
    Path('/kaggle/input/godzilla-tokenized-15k/tokenized'),
    Path('/kaggle/input/godzilla-tokenized-15k'),
    Path('/kaggle/input/godzilla-piano-tokenized/tokenized'),
    Path('/kaggle/input/godzilla-piano-tokenized'),
]

def find_resume_checkpoint(checkpoint_dir: Path) -> str:
    candidates = [
        checkpoint_dir / 'latest_state.pt',
        checkpoint_dir / 'latest.safetensors',
        checkpoint_dir / 'best_state.pt',
        checkpoint_dir / 'best.safetensors',
    ]
    for candidate in candidates:
        if candidate.exists():
            return str(candidate)
    return ''

def discover_npz_parent(path: Path):
    if not path.exists():
        return None
    hit = next(path.rglob('*.npz'), None)
    if hit is None:
        return None
    return hit.parent

def find_npz_root(candidates) -> Path:
    for candidate in candidates:
        resolved = discover_npz_parent(Path(candidate))
        if resolved is not None:
            return resolved

    kaggle_input = Path('/kaggle/input')
    if kaggle_input.exists():
        for folder in kaggle_input.iterdir():
            if not folder.is_dir():
                continue
            resolved = discover_npz_parent(folder)
            if resolved is not None:
                return resolved

    raise FileNotFoundError(
        'Could not locate tokenized NPZ files. Set PRETOKENIZED_ROOT_CANDIDATES to your dataset path.'
    )

def build_manifest_from_npz(npz_root: Path, manifest_path: Path) -> int:
    npz_files = sorted(npz_root.rglob('*.npz'))
    if not npz_files:
        raise FileNotFoundError(f'No .npz files found under: {npz_root}')

    manifest = []
    skipped = 0
    for npz_path in npz_files:
        try:
            with np.load(npz_path, allow_pickle=False) as pack:
                if 'tokens' not in pack.files:
                    skipped += 1
                    continue
                token_len = int(np.asarray(pack['tokens']).shape[0])
        except Exception:
            skipped += 1
            continue

        manifest.append({
            'md5': npz_path.stem,
            'npz_path': str(npz_path.resolve()),
            'length': token_len,
            'source_path': '',
        })

    manifest_path.parent.mkdir(parents=True, exist_ok=True)
    manifest_path.write_text(json.dumps(manifest, indent=2), encoding='utf-8')
    print(f'Manifest built: {manifest_path} | entries={len(manifest)} | skipped={skipped}')
    return len(manifest)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PRETOKENIZED_ROOT = find_npz_root(PRETOKENIZED_ROOT_CANDIDATES)
PRETOKENIZED_MANIFEST = OUTPUT_DIR / 'processed_pretokenized' / 'manifest.json'
if FORCE_REBUILD_MANIFEST or not PRETOKENIZED_MANIFEST.exists():
    _ = build_manifest_from_npz(PRETOKENIZED_ROOT, PRETOKENIZED_MANIFEST)
else:
    print(f'Using existing manifest: {PRETOKENIZED_MANIFEST}')

checkpoint_dir = OUTPUT_DIR / 'checkpoints' / f'variant_e_{SIZE_PRESET}'
resume_ckpt = find_resume_checkpoint(checkpoint_dir) if AUTO_RESUME else ''
RESUME_FROM_CHECKPOINT = resume_ckpt if resume_ckpt else ''
print(f'Pretokenized root: {PRETOKENIZED_ROOT}')
print(f'Manifest: {PRETOKENIZED_MANIFEST}')
print(f'Output base: {OUTPUT_BASE}')
print(f'Output: {OUTPUT_DIR}')
print(f'Resume checkpoint: {resume_ckpt or "none"}')

In [ ]:
cmd = [
    sys.executable, '-m', TRAIN_MODULE,
    '--size_preset', SIZE_PRESET,
    '--pretokenized_manifest', str(PRETOKENIZED_MANIFEST),
    '--pretokenized_root', str(PRETOKENIZED_ROOT),
    '--output_dir', str(OUTPUT_DIR),
    '--max_pieces', str(MAX_PIECES),
    '--epochs', str(EPOCHS),
    '--batch_size', str(BATCH_SIZE),
    '--grad_accumulation_steps', str(GRAD_ACCUM_STEPS),
    '--num_workers', str(NUM_WORKERS),
    '--learning_rate', str(LEARNING_RATE),
    '--seed', str(SEED),
    '--log_every_n_steps', str(LOG_EVERY_N_STEPS),
]
if RESUME_FROM_CHECKPOINT:
    cmd.extend(['--resume_from_checkpoint', RESUME_FROM_CHECKPOINT, '--resume_mode', RESUME_MODE])
if ALLOW_FALLBACK_GDN:
    cmd.append('--allow_fallback_gdn')
if str(SEED_MIDI).strip():
    cmd.extend(['--seed_midi', str(SEED_MIDI)])
print(shlex.join(cmd))

In [ ]:
RUN_TRAINING = False
if RUN_TRAINING:
    if not PRETOKENIZED_MANIFEST.exists():
        raise FileNotFoundError(f'Manifest not found: {PRETOKENIZED_MANIFEST}')
    run_env = os.environ.copy()
    run_env['PYTHONPATH'] = str(PROJECT_ROOT) + os.pathsep + run_env.get('PYTHONPATH', '')
    subprocess.run(cmd, cwd=str(PROJECT_ROOT), env=run_env, check=True)
else:
    print('Dry run only. Set RUN_TRAINING=True to launch.')

result_path = OUTPUT_DIR / f'variant_e_{SIZE_PRESET}_result.json'
if result_path.exists():
    payload = json.loads(result_path.read_text(encoding='utf-8'))
    print('Result path:', result_path)
    print('Params:', payload.get('result', {}).get('params'))
    val_loss = payload.get('result', {}).get('val_loss', [])
    if val_loss:
        print('Last val loss:', val_loss[-1])
    print('Resume metadata:', payload.get('result', {}).get('resume', {}))
    print('Backend status:', payload.get('backend_status', {}))
else:
    print('No result JSON yet.')